# ETL_geocode_bairros_osm

This notebook creates a **draft geocoding table** for the neighborhoods in `dim_bairro` using Nominatim/OpenStreetMap.

The notebook does **not** change `dim_bairro`. It only creates geocoding support tables.

## 1. Setup

The bounding box below is an approximate safety filter around Araçatuba/SP. It is used only to flag obviously wrong coordinates.

In [0]:
# =========================
# 1. Setup
# =========================

import requests
import time
from datetime import datetime

from pyspark.sql.functions import (
    col, lit, when, coalesce, count as spark_count
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType, BooleanType
)

CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

GEOCODING_INPUT_TABLE = f"{TABLE_PREFIX}.ref_bairros_para_geocoding"
GEOCODING_DRAFT_TABLE = f"{TABLE_PREFIX}.bairro_geocoding_draft"
GEOCODING_VALIDATED_TABLE = f"{TABLE_PREFIX}.bairro_geocoding_validated"

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "epidemiologia-aracatuba-univesp/1.0"
REQUEST_DELAY_SECONDS = 1.2

MIN_LAT = -21.35
MAX_LAT = -21.05
MIN_LON = -50.65
MAX_LON = -50.25

APPROVE_AUTOMATIC_GEOCODING = False

## 2. Load neighborhoods from `dim_bairro`

In [0]:
# =========================
# 2. Load standardized neighborhoods for geocoding
# =========================

GEOCODING_INPUT_TABLE = f"{TABLE_PREFIX}.ref_bairros_para_geocoding"

df_bairros = (
    spark.table(GEOCODING_INPUT_TABLE)
    .select(
        col("id_bairro_canonico"),
        col("bairro_canonico").alias("bairro"),
        col("query_geocoding")
    )
    .orderBy("bairro")
)

print(f"Standardized neighborhoods loaded for geocoding: {df_bairros.count()}")
display(df_bairros)

## 3. Geocoding helper functions

The function tries a few query formats and keeps the best candidate. Every result is still marked for review.

In [0]:
# =========================
# 3. Geocoding helper functions
# =========================

def is_inside_aracatuba_bbox(latitude, longitude):
    if latitude is None or longitude is None:
        return False

    return (
        MIN_LAT <= latitude <= MAX_LAT and
        MIN_LON <= longitude <= MAX_LON
    )


def score_candidate(item):
    display_name = (item.get("display_name") or "").lower()

    try:
        latitude = float(item.get("lat"))
        longitude = float(item.get("lon"))
    except Exception:
        return -1

    score = 0

    if is_inside_aracatuba_bbox(latitude, longitude):
        score += 5

    if "araçatuba" in display_name or "aracatuba" in display_name:
        score += 3

    if "são paulo" in display_name or "sao paulo" in display_name:
        score += 1

    # Prefer neighborhood/suburb-like results when available.
    osm_class = (item.get("class") or "").lower()
    osm_type = (item.get("type") or "").lower()

    if osm_type in ["neighbourhood", "suburb", "quarter", "residential"]:
        score += 2

    if osm_class in ["place", "boundary"]:
        score += 1

    return score


def request_nominatim(query):
    params = {
        "q": query,
        "format": "json",
        "limit": 5,
        "addressdetails": 1,
        "extratags": 1,
    }

    headers = {"User-Agent": USER_AGENT}

    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()

    # Respect Nominatim public usage policy.
    time.sleep(REQUEST_DELAY_SECONDS)

    return response.json()


def geocode_neighborhood(bairro, query_geocoding=None):
    if query_geocoding:
        queries = [
            query_geocoding,
            f"{bairro}, Araçatuba, São Paulo, Brasil",
            f"Bairro {bairro}, Araçatuba, São Paulo, Brasil",
            f"{bairro}, Araçatuba, SP, Brazil",
        ]
    else:
        queries = [
            f"{bairro}, Araçatuba, São Paulo, Brasil",
            f"Bairro {bairro}, Araçatuba, São Paulo, Brasil",
            f"{bairro}, Araçatuba, SP, Brazil",
        ]

    candidates = []
    used_query = None

    for query in queries:
        data = request_nominatim(query)
        if data:
            candidates.extend(data)
            used_query = query

    if not candidates:
        return {
            "bairro": bairro,
            "query_used": None,
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_class": None,
            "osm_type": None,
            "osm_id": None,
            "candidate_score": 0,
            "is_inside_aracatuba_bbox": False,
            "geocode_status": "not_found",
            "needs_review": True,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        }

    best = sorted(candidates, key=score_candidate, reverse=True)[0]

    try:
        latitude = float(best.get("lat"))
        longitude = float(best.get("lon"))
    except Exception:
        latitude = None
        longitude = None

    inside_bbox = is_inside_aracatuba_bbox(latitude, longitude)
    candidate_score = score_candidate(best)

    status = "found"
    if not inside_bbox:
        status = "found_outside_bbox"

    return {
        "bairro": bairro,
        "query_used": used_query,
        "latitude": latitude,
        "longitude": longitude,
        "display_name": best.get("display_name"),
        "osm_class": best.get("class"),
        "osm_type": best.get("type"),
        "osm_id": str(best.get("osm_id")) if best.get("osm_id") is not None else None,
        "candidate_score": candidate_score,
        "is_inside_aracatuba_bbox": inside_bbox,
        "geocode_status": status,
        "needs_review": True,
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }

## 4. Run geocoding

This step may take a few minutes because requests are intentionally slowed down.

In [0]:
# =========================
# 4. Run geocoding
# =========================

bairro_rows = df_bairros.collect()
geo_rows = []

for row in bairro_rows:
    id_bairro_canonico = row["id_bairro_canonico"]
    bairro = row["bairro"]
    query_geocoding = row["query_geocoding"]

    try:
        result = geocode_neighborhood(bairro, query_geocoding)
        result["id_bairro_canonico"] = id_bairro_canonico
        geo_rows.append(result)

        print(
            f"OK: {bairro} -> {result['latitude']}, {result['longitude']} | "
            f"status={result['geocode_status']} | score={result['candidate_score']}"
        )

    except Exception as e:
        geo_rows.append({
            "id_bairro_canonico": id_bairro_canonico,
            "bairro": bairro,
            "query_used": query_geocoding,
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "osm_class": None,
            "osm_type": None,
            "osm_id": None,
            "candidate_score": 0,
            "is_inside_aracatuba_bbox": False,
            "geocode_status": f"error: {str(e)}",
            "needs_review": True,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        })

        print(f"ERROR: {bairro} -> {e}")


schema_geo = StructType([
    StructField("id_bairro_canonico", IntegerType(), False),
    StructField("bairro", StringType(), False),
    StructField("query_used", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("display_name", StringType(), True),
    StructField("osm_class", StringType(), True),
    StructField("osm_type", StringType(), True),
    StructField("osm_id", StringType(), True),
    StructField("candidate_score", IntegerType(), True),
    StructField("is_inside_aracatuba_bbox", BooleanType(), True),
    StructField("geocode_status", StringType(), True),
    StructField("needs_review", BooleanType(), True),
    StructField("created_at", StringType(), True),
])

## 5. Review problematic records

Check these rows carefully. They usually need manual correction.

In [0]:
# =========================
# 5. Review problematic records
# =========================

df_review = spark.table(GEOCODING_DRAFT_TABLE)

problem_filter = (
    col("latitude").isNull() |
    col("longitude").isNull() |
    (col("is_inside_aracatuba_bbox") == False) |
    (col("candidate_score") < 5)
)

display(
    df_review
    .filter(problem_filter)
    .select(
        "id_bairro_canonico",
        "bairro",
        "latitude",
        "longitude",
        "display_name",
        "candidate_score",
        "is_inside_aracatuba_bbox",
        "geocode_status"
    )
    .orderBy("bairro")
)

## 6. Optional manual corrections

Use this list to correct missing or wrong coordinates after reviewing the draft table.

Leave the list empty if no correction is needed.

In [0]:
# =========================
# 6. Optional manual corrections
# =========================

# Format:
# manual_corrections = [
#     ("bairro name exactly as in bairro_geocoding_draft", latitude, longitude, "manual_source_or_note"),
# ]

manual_corrections = [
    # Example only. Replace with real values if needed.
    # ("Centro", -21.1855268, -50.4531115, "manual review"),
]

schema_manual = StructType([
    StructField("bairro", StringType(), False),
    StructField("manual_latitude", DoubleType(), True),
    StructField("manual_longitude", DoubleType(), True),
    StructField("manual_source", StringType(), True),
])

df_manual = spark.createDataFrame(manual_corrections, schema=schema_manual)

df_geo_corrected = (
    spark.table(GEOCODING_DRAFT_TABLE)
    .join(df_manual, on="bairro", how="left")
    .withColumn("latitude_final", coalesce(col("manual_latitude"), col("latitude")))
    .withColumn("longitude_final", coalesce(col("manual_longitude"), col("longitude")))
    .withColumn(
        "is_inside_aracatuba_bbox_final",
        (col("latitude_final") >= lit(MIN_LAT)) &
        (col("latitude_final") <= lit(MAX_LAT)) &
        (col("longitude_final") >= lit(MIN_LON)) &
        (col("longitude_final") <= lit(MAX_LON))
    )
    .withColumn(
        "geocode_source_final",
        when(
            col("manual_latitude").isNotNull() & col("manual_longitude").isNotNull(),
            col("manual_source")
        ).otherwise(lit("nominatim_osm"))
    )
    .withColumn(
        "geocode_status_final",
        when(col("latitude_final").isNull() | col("longitude_final").isNull(), lit("not_found"))
        .when(col("is_inside_aracatuba_bbox_final") == False, lit("found_outside_bbox"))
        .otherwise(lit("found"))
    )
    .withColumn(
        "needs_review_final",
        when(
            col("latitude_final").isNull() |
            col("longitude_final").isNull() |
            (col("is_inside_aracatuba_bbox_final") == False) |
            (col("candidate_score") < 5),
            lit(True)
        ).otherwise(lit(False))
    )
)

display(
    df_geo_corrected
    .select(
        "id_bairro_canonico",
        "bairro",
        "latitude_final",
        "longitude_final",
        "display_name",
        "geocode_status_final",
        "candidate_score",
        "is_inside_aracatuba_bbox_final",
        "geocode_source_final",
        "needs_review_final"
    )
    .orderBy("bairro")
)

## 7. Create validated geocoding table

By default, this step does **not** write the validated table. Set `APPROVE_AUTOMATIC_GEOCODING = True` in the setup cell only after reviewing the coordinates.

In [0]:
# =========================
# 7. Create validated geocoding table
# =========================

if APPROVE_AUTOMATIC_GEOCODING:
    df_validated = (
        df_geo_corrected
        .filter(col("latitude_final").isNotNull())
        .filter(col("longitude_final").isNotNull())
        .filter(col("is_inside_aracatuba_bbox_final") == True)
        .select(
            "id_bairro_canonico",
            "bairro",
            col("latitude_final").alias("latitude"),
            col("longitude_final").alias("longitude"),
            "display_name",
            "osm_class",
            "osm_type",
            "osm_id",
            "candidate_score",
            col("is_inside_aracatuba_bbox_final").alias("is_inside_aracatuba_bbox"),
            col("geocode_status_final").alias("geocode_status"),
            "geocode_source_final",
            "geocoded_at"
        )
        .orderBy("id_bairro_canonico")
    )

    df_validated.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GEOCODING_VALIDATED_TABLE)

    print(f"Validated table saved as: {GEOCODING_VALIDATED_TABLE}")
    display(df_validated)

else:
    print("Validated table was NOT created.")
    print("Review bairro_geocoding_draft first. Then set APPROVE_AUTOMATIC_GEOCODING = True and rerun this cell.")

## 8. Summary checks

In [0]:
# =========================
# 8. Summary checks
# =========================

print("Draft geocoding summary:")

df_draft = spark.table(GEOCODING_DRAFT_TABLE)

df_draft \
    .groupBy("geocode_status") \
    .count() \
    .orderBy("geocode_status") \
    .show(truncate=False)

print("Records without coordinates:")

df_draft \
    .filter(col("latitude").isNull() | col("longitude").isNull()) \
    .select(
        "id_bairro_canonico",
        "bairro",
        "geocode_status",
        "candidate_score"
    ) \
    .orderBy("bairro") \
    .show(200, truncate=False)

print("Records with coordinates outside the approximate Araçatuba bounding box:")

df_draft \
    .filter(
        col("latitude").isNotNull() &
        col("longitude").isNotNull() &
        (col("is_inside_aracatuba_bbox") == False)
    ) \
    .select(
        "id_bairro_canonico",
        "bairro",
        "latitude",
        "longitude",
        "display_name",
        "geocode_status"
    ) \
    .orderBy("bairro") \
    .show(200, truncate=False)